In [1]:
"""
Initializes visualization settings, loads the dataset, cleans column names,
and filters for valid target classes.
"""
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('../data/data.csv', sep=';', engine='python')
df.columns = [c.strip().strip('"').replace("\t", "").strip() for c in df.columns]
df = df[df["Target"].isin(["Graduate", "Dropout"])].copy()

df.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


بررسی اطلاعات کلی دیتاست و چک کردن مقادیر Null (گمشده).

In [2]:
"""
Displays basic dataset information and counts total missing values.
"""
print(df.info())
print("\n" + "="*40)
print(f"Total Missing Values in Dataset: {df.isnull().sum().sum()}")

<class 'pandas.DataFrame'>
Index: 3630 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  3630 non-null   int64  
 1   Application mode                                3630 non-null   int64  
 2   Application order                               3630 non-null   int64  
 3   Course                                          3630 non-null   int64  
 4   Daytime/evening attendance                      3630 non-null   int64  
 5   Previous qualification                          3630 non-null   int64  
 6   Previous qualification (grade)                  3630 non-null   float64
 7   Nacionality                                     3630 non-null   int64  
 8   Mother's qualification                          3630 non-null   int64  
 9   Father's qualification                          3630 non-

> **Insight 1: Data Quality & Completeness**
> The dataset consists of 3,630 records and 37 features. Notably, there are zero missing values. This eliminates the need for complex imputation techniques and ensures high-quality data for model training.

بررسی توزیع متغیر هدف (Target). این نمودار به ما نشان می‌دهد که آیا با مشکل Data Imbalance (عدم توازن کلاس‌ها) مواجه هستیم یا خیر.

In [ ]:
"""
Plots the distribution of the target classes and annotates each bar
with its corresponding percentage for better readability.
"""
plt.figure(figsize=(8, 5))

# آپدیت برای رفع وارنینگ: متغیر x به hue هم داده شد و legend غیرفعال شد
ax = sns.countplot(data=df, x='Target', hue='Target', palette='Set2', legend=False)

plt.title('Target Class Distribution: Graduate vs Dropout', fontweight='bold', pad=15)

total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2 - 0.1
    y = p.get_y() + p.get_height() + 10
    ax.annotate(percentage, (x, y), fontweight='bold')

plt.tight_layout()
plt.show()

> **Insight 2: Class Distribution & Imbalance**
> The target variable exhibits a moderate class imbalance (60.9% Graduate vs. 39.1% Dropout). While this doesn't necessitate aggressive sampling techniques like SMOTE, it justifies the use of `scale_pos_weight` in XGBoost and `Stratified K-Fold` cross-validation to prevent model bias toward the majority class.

بررسی همبستگی (Correlation) بین ویژگی‌های عددی دیتاست برای پیدا کردن متغیرهایی که بیشترین ارتباط خطی را با هم دارند.

In [ ]:
"""
Extracts numeric features, calculates the Pearson correlation matrix,
and visualizes it using a heatmap without annotations to prevent clutter.
"""
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

plt.figure(figsize=(16, 12))
corr = df[numeric_cols].corr()

sns.heatmap(
    corr, 
    cmap='coolwarm', 
    center=0, 
    annot=False, 
    linewidths=0.5
)

plt.title('Correlation Matrix of Numeric Features', fontweight='bold', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

> **Insight 3: Multicollinearity & Feature Relationships**
> The heatmap reveals strong positive linear correlations among academic performance metrics across semesters (e.g., enrolled units, approved units, and grades). This severe multicollinearity directly validates our rigorous Feature Selection pipeline (combining Mutual Information and L1 Regularization) to eliminate redundant features and mitigate the risk of overfitting.